In [1]:
import polars as pl
from Python import identity, batter_features, batter_rolling, statcast

years = [2025]

In [2]:
print(batter_rolling.DEFAULT_WINDOWS)
print(batter_rolling.DEFAULT_SHRINK_PA, batter_rolling.DEFAULT_FALLBACK_K_RATE)
print(batter_rolling.DEFAULT_EXTRA_RATE_STATS)

(5, 10, 20)
200.0 None
{'whiff_rate': ('Whiffs', 'Pitches'), 'chase_rate': ('Chases', 'OutZone')}


In [3]:
columns = ["game_year", *batter_features.BUILD_COLUMNS]
raw = statcast.load_statcast_years(years, columns=columns)

_, _, official_game_pks = statcast.regular_season_schedule(years[0])
statcast.validate_statcast_season(raw, years[0], official_game_pks=official_game_pks)

# Load the immediately preceding season only to source the first-date league-K prior.
prior_raw = statcast.load_statcast_years(
    [min(years) - 1],
    columns=["game_pk", "game_date", "game_year", "at_bat_number", "pitch_number", "events"],
)
prior_pa = statcast.plate_appearances(prior_raw)
prior_league_k_rate = float(prior_pa["is_k"].sum() / prior_pa.height)

games = batter_features.build_batter_games(raw).with_columns(
    pl.lit(prior_league_k_rate).alias("prior_league_k_rate")
)
games = identity.enrich_batter_names(games)
games.shape

(48862, 46)

In [4]:
unmapped_batters = identity.unmapped_player_ids(
    games, id_column="batter", name_column="batter_name"
)
print("Unmapped batter IDs:", len(unmapped_batters))
print("Sample:", unmapped_batters[:10])

Unmapped batter IDs: 0
Sample: ()


In [5]:
games.select(["game_pk", "batter"]).is_duplicated().sum()

0

In [6]:
required_cols = ["batter", "game_date", "game_pk", "PA", "K", "PA_vL", "K_vL", "PA_vR", "K_vR"]
missing_required = [c for c in required_cols if c not in games.columns]

missing_extras = {
    name: (num, den) for name, (num, den) in batter_rolling.DEFAULT_EXTRA_RATE_STATS.items()
    if num not in games.columns or den not in games.columns
}

print("Missing required columns:", missing_required)
print("Extra rate stats skipped (missing num/den):", missing_extras)

Missing required columns: []
Extra rate stats skipped (missing num/den): {}


In [7]:
rolled = batter_rolling.add_leakage_safe_k(games)
rolled.shape

# Uniqueness check on the primary key -- should still be 0 duplicates after adding columns
rolled.select(["game_pk", "batter"]).is_duplicated().sum()

0

In [8]:
new_cols = [c for c in rolled.columns if c not in games.columns]
print(len(new_cols), "new columns added")
new_cols

10 new columns added


['season',
 'k_rate_std',
 'k_rate_std_vL',
 'k_rate_std_vR',
 'whiff_rate_std',
 'chase_rate_std',
 'k_rate_P5',
 'k_rate_P10',
 'k_rate_P20',
 'k_rate_std_shrunk']

In [9]:
ordered = rolled.sort(["batter", "game_date", "game_pk"])
season_first = ordered.group_by(["batter", "season"]).first()
career_first = ordered.group_by("batter").first()

# Season-to-date resets every season. Rolling form carries across seasons, so it
# is required to be null only on a batter's first game in the loaded history.
print(
    "STD null on every season opener:",
    season_first["k_rate_std"].is_null().all(),
)
print(
    "P5 null on every career-first row:",
    career_first["k_rate_P5"].is_null().all(),
)

STD null on every season opener: True
P5 null on every career-first row: True


In [10]:
sample_batter = rolled["batter"][0]

manual_check = (
    rolled.filter(pl.col("batter") == sample_batter)
    .sort("game_date")
    .select(["game_date", "K", "PA", "k_rate_P5", "k_rate_std"])
    .head(8)
)
manual_check

# Manually verify: row N's k_rate_P5 should equal sum(K)/sum(PA) over the
# PRECEDING up-to-5 rows only, never including row N's own K/PA.

game_date,K,PA,k_rate_P5,k_rate_std
date,u32,u32,f64,f64
2025-03-27,1,1,null,null
2025-03-28,1,3,1.0,1.0
2025-03-29,1,3,0.5,0.5
2025-04-02,1,4,0.428571,0.428571
2025-04-04,0,2,0.363636,0.363636
2025-04-08,0,4,0.307692,0.307692
2025-04-09,3,4,0.1875,0.235294
2025-04-12,0,3,0.294118,0.333333


In [11]:
rolled.filter(pl.col("batter") == sample_batter).sort("game_date").select(
    ["game_date", "PA_vL", "K_vL", "k_rate_std_vL", "PA_vR", "K_vR", "k_rate_std_vR"]
).head(8)

game_date,PA_vL,K_vL,k_rate_std_vL,PA_vR,K_vR,k_rate_std_vR
date,u32,u32,f64,u32,u32,f64
2025-03-27,0,0,null,1,1,null
2025-03-28,0,0,null,3,1,1.0
2025-03-29,0,0,null,3,1,0.5
2025-04-02,0,0,null,4,1,0.428571
2025-04-04,2,0,null,0,0,0.363636
2025-04-08,3,0,0.0,1,0,0.363636
2025-04-09,1,1,0.0,3,2,0.333333
2025-04-12,0,0,0.166667,3,0,0.4


In [12]:
shrink_check = (
    rolled.filter(pl.col("batter") == sample_batter)
    .sort("game_date")
    .select(["game_date", "k_rate_std", "k_rate_std_shrunk"])
    .head(6)
)
shrink_check

# Early games: k_rate_std_shrunk should sit closer to the sourced prior-season
# league K rate than raw k_rate_std because prior PA is small vs shrink_pa=200.

game_date,k_rate_std,k_rate_std_shrunk
date,f64,f64
2025-03-27,null,0.254902
2025-03-28,1.0,0.250844
2025-03-29,0.5,0.247937
2025-04-02,0.428571,0.233358
2025-04-04,0.363636,0.23795
2025-04-08,0.307692,0.229647


In [13]:
rolled.select(["k_rate_std", "k_rate_std_shrunk", "whiff_rate_std", "chase_rate_std"]).describe()

statistic,k_rate_std,k_rate_std_shrunk,whiff_rate_std,chase_rate_std
str,f64,f64,f64,f64
"""count""",48186.0,48862.0,48186.0,48168.0
"""null_count""",676.0,0.0,676.0,694.0
"""mean""",0.222611,0.218797,0.120565,0.282388
"""std""",0.085583,0.030076,0.043569,0.074304
"""min""",0.0,0.07472,0.0,0.0
"""25%""",0.169591,0.201355,0.092784,0.235294
"""50%""",0.221719,0.222096,0.117936,0.276398
"""75%""",0.264977,0.238133,0.144444,0.323232
"""max""",1.0,0.323371,1.0,1.0


In [14]:
season_boundary = (
    rolled.filter(pl.col("batter") == sample_batter)
    .sort("game_date")
    .select(["game_date", "season", "k_rate_std"])
)

season_boundary.with_columns(
    (pl.col("season") != pl.col("season").shift(1)).alias("season_changed")
).filter(pl.col("season_changed"))

game_date,season,k_rate_std,season_changed
date,i32,f64,bool


In [15]:
zero_pa_rows = rolled.filter(pl.col("PA") == 0)
zero_pa_rows.height

# Confirm these rows don't poison k_rate_std/k_rate_std_shrunk for the batter's
# NEXT game -- prior_num/prior_den simply add 0, which is harmless by design.
zero_pa_rows.select(["batter", "game_date", "k_rate_std", "k_rate_std_shrunk"])

batter,game_date,k_rate_std,k_rate_std_shrunk
i64,date,f64,f64
642350,2025-04-12,0.333333,0.236351
670764,2025-06-01,0.201342,0.211462
672515,2025-04-20,0.172414,0.212991
696285,2025-07-26,0.12749,0.167987
